# CEG-WM Colab Mini-Real Validation Notebook

本 Notebook 用于在 Google Colab 上执行真实 Stable Diffusion 3.5 Medium 模型的 mini-real validation。

目标不是生成论文级最终指标，而是验证 formal matrix 的三个工程前置条件是否真实成立：
- real negative cache 真实生成
- global calibrate 与 shared thresholds 真实生成
- forced pair fallback 在 formal guard 下被显式阻断

本 Notebook 直接调用 scripts/run_mini_real_validation.py，而不是在 Notebook 中复制正式主链。

主要输出：
- artifacts/workflow_acceptance/paper_full_mini_real_validation_summary.json
- artifacts/run_closure.json
- outputs/experiment_matrix/artifacts/grid_summary.json
- logs/ 下的完整执行日志

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import psutil

try:
    from google.colab import drive
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = False
COLAB_ROOT = Path('/content') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG_WM_Colab')
WORK_DIR = COLAB_ROOT
PERSISTENT_ROOT = DRIVE_ROOT if USE_GOOGLE_DRIVE and IN_COLAB else (WORK_DIR / 'ceg_wm_runtime')
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

print('=' * 80)
print('初始化 Colab mini-real validation 环境')
print('=' * 80)
print(f'IN_COLAB={IN_COLAB}')
print(f'WORK_DIR={WORK_DIR}')
print(f'PERSISTENT_ROOT={PERSISTENT_ROOT}')
print(f'Python={sys.version.split()[0]}')
print(f'CPU cores={psutil.cpu_count()}')
print(f'RAM total={psutil.virtual_memory().total / (1024 ** 3):.1f} GB')
print(f'RAM available={psutil.virtual_memory().available / (1024 ** 3):.1f} GB')

In [ ]:
print('=' * 80)
print('挂载 Google Drive')
print('=' * 80)

GOOGLE_DRIVE_ROOT = None
GOOGLE_DRIVE_EXPORT_DIR = None

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount('/content/drive', force_remount=False)
        GOOGLE_DRIVE_ROOT = Path('/content/drive/MyDrive')
        GOOGLE_DRIVE_EXPORT_DIR = GOOGLE_DRIVE_ROOT / 'CEG_WM_Outputs' / 'Mini_Real_Validation'
        GOOGLE_DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f'✓ Google Drive 已挂载: {GOOGLE_DRIVE_ROOT}')
        print(f'✓ 导出同步目录: {GOOGLE_DRIVE_EXPORT_DIR}')

        if USE_GOOGLE_DRIVE:
            PERSISTENT_ROOT = DRIVE_ROOT
            PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
            print(f'✓ 已启用 Google Drive 持久化目录: {PERSISTENT_ROOT}')
    except Exception as exc:
        GOOGLE_DRIVE_ROOT = None
        GOOGLE_DRIVE_EXPORT_DIR = None
        print(f'✗ Google Drive 挂载失败: {exc}')
else:
    print('ℹ 当前不是 Google Colab 环境，跳过 Google Drive 挂载')

## 第 1 步：同步仓库代码

默认沿用你当前 Notebook 的仓库地址与分支。Colab 每次重连后建议重新执行本步，确保脚本入口和配置文件与当前仓库一致。

In [ ]:
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
REPO_BRANCH = 'main'
TARGET_DIR = WORK_DIR / 'CEG-WM'

def run_checked(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'command failed: {command}\nstdout_tail={result.stdout[-800:]}\nstderr_tail={result.stderr[-800:]}'
        )
    return result

if shutil.which('git') is None:
    raise RuntimeError('当前运行时未检测到 git，请先安装系统依赖。')

if TARGET_DIR.exists() and (TARGET_DIR / '.git').exists():
    print(f'复用现有仓库: {TARGET_DIR}')
    run_checked(['git', '-C', str(TARGET_DIR), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'])
    run_checked(['git', '-C', str(TARGET_DIR), 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'])
    run_checked(['git', '-C', str(TARGET_DIR), 'reset', '--hard', 'FETCH_HEAD'])
    run_checked(['git', '-C', str(TARGET_DIR), 'clean', '-fd'])
else:
    if TARGET_DIR.exists():
        shutil.rmtree(TARGET_DIR)
    run_checked(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(TARGET_DIR)])

CEG_WM_ROOT = TARGET_DIR
if str(CEG_WM_ROOT) not in sys.path:
    sys.path.insert(0, str(CEG_WM_ROOT))

required_paths = [
    CEG_WM_ROOT / 'scripts' / 'run_mini_real_validation.py',
    CEG_WM_ROOT / 'configs' / 'paper_full_cuda_mini_real_validation.yaml',
    CEG_WM_ROOT / 'prompts' / 'paper_small_mini_real.txt',
    CEG_WM_ROOT / 'requirements.txt',
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'缺失 mini-real validation 关键文件: {missing}')

print(f'✓ CEG_WM_ROOT={CEG_WM_ROOT}')
print('✓ mini-real validation 关键文件已就绪')

## 第 2 步：安装依赖

这一版保留你当前 Colab Workflow 的安装习惯，但把目标固定到 mini-real validation 所需依赖。

In [ ]:
print('=' * 80)
print('安装系统依赖与 Python 依赖')
print('=' * 80)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=False)
if IN_COLAB:
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git', 'wget', 'unzip'], check=False)

editable_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(CEG_WM_ROOT)],
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
)
if editable_result.returncode != 0:
    print(editable_result.stdout[-500:])
    print(editable_result.stderr[-500:])
    raise RuntimeError('pip install -e failed')

extra_packages = [
    'transparent-background',
    'huggingface_hub',
    'ipywidgets',
]
extra_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + extra_packages,
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
)
if extra_result.returncode != 0:
    print(extra_result.stdout[-500:])
    print(extra_result.stderr[-500:])
    raise RuntimeError('extra package installation failed')

print('✓ 依赖安装完成')

## 第 3 步：Hugging Face 认证与模型准备

真实 SD3.5 运行依赖 Hugging Face 访问权限。你可以直接在本 cell 填写 HF_TOKEN，也可以把 USE_NOTEBOOK_LOGIN 改成 True 后走交互登录。

In [ ]:
import urllib.request

from huggingface_hub import HfApi, login, notebook_login, scan_cache_dir, snapshot_download

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
USE_NOTEBOOK_LOGIN = False
MODEL_ID = 'stabilityai/stable-diffusion-3.5-medium'
MODEL_CACHE_DIR = PERSISTENT_ROOT / 'huggingface_cache'
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(MODEL_CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(MODEL_CACHE_DIR)

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ 已使用 HF_TOKEN 登录')
elif USE_NOTEBOOK_LOGIN:
    notebook_login()
else:
    print('ℹ 当前未显式登录 Hugging Face；若模型访问失败，请设置 HF_TOKEN 或开启 notebook_login。')

api = HfApi()
try:
    _ = api.model_info(MODEL_ID)
    print(f'✓ 模型可访问: {MODEL_ID}')
except Exception as exc:
    raise RuntimeError(f'无法访问 {MODEL_ID}: {type(exc).__name__}: {exc}')

def model_cached(model_id: str) -> bool:
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if model_id in repo.repo_id:
                return True
    except Exception:
        return False
    return False

if model_cached(MODEL_ID):
    print('✓ SD3.5 已缓存，跳过下载')
else:
    print('开始下载 SD3.5 模型，这一步可能需要数分钟...')
    snapshot_path = snapshot_download(
        repo_id=MODEL_ID,
        cache_dir=str(MODEL_CACHE_DIR),
        local_files_only=False,
    )
    print(f'✓ 模型下载完成: {snapshot_path}')

inspyrenet_target = CEG_WM_ROOT / 'models' / 'inspyrenet' / 'ckpt_base.pth'
inspyrenet_target.parent.mkdir(parents=True, exist_ok=True)
if not inspyrenet_target.exists() or inspyrenet_target.stat().st_size <= 0:
    url = 'https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth'
    print(f'下载 InSPyReNet 权重: {url}')
    urllib.request.urlretrieve(url, str(inspyrenet_target))

if not inspyrenet_target.exists() or inspyrenet_target.stat().st_size <= 0:
    raise RuntimeError(f'InSPyReNet 权重无效: {inspyrenet_target}')

print(f'✓ InSPyReNet 权重就绪: {inspyrenet_target}')
gc.collect()
if 'torch' in globals() and hasattr(torch, 'cuda') and torch.cuda.is_available():
    torch.cuda.empty_cache()

## 第 4 步：配置选择与 attestation 环境变量

这里固定使用 mini-real validation 配置，并为 formal 路径准备 attestation 环境变量。默认会为缺失项生成当前会话有效的临时密钥。

In [ ]:
import re
import secrets

CONFIG_CHOICE = 'paper_full_cuda_mini_real_validation'
CONFIG_FILE = CEG_WM_ROOT / 'configs' / f'{CONFIG_CHOICE}.yaml'
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f'配置文件不存在: {CONFIG_FILE}')

ATTESTATION_ENV_SPECS = {
    'CEG_WM_K_MASTER': 64,
    'CEG_WM_K_PROMPT': 32,
    'CEG_WM_K_SEED': 32,
}
USER_ATTESTATION_VALUES = {
    'CEG_WM_K_MASTER': '',
    'CEG_WM_K_PROMPT': '',
    'CEG_WM_K_SEED': '',
}
AUTO_GENERATE_MISSING_ATTESTATION_KEYS = True
ATTESTATION_INFO_PATH = WORK_DIR / 'attestation_env_keys.json'

def is_valid_hex_secret(value: str, expected_length: int) -> bool:
    if not isinstance(value, str):
        return False
    candidate = value.strip().lower()
    return len(candidate) == expected_length and re.fullmatch(r'[0-9a-f]+', candidate) is not None

def mask_secret(value: str) -> str:
    if len(value) <= 8:
        return '*' * len(value)
    return f'{value[:4]}...{value[-4:]}'

resolved_values = {}
generated_env_vars = []
invalid_env_vars = []
for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    manual_value = USER_ATTESTATION_VALUES.get(env_name, '')
    existing_value = os.environ.get(env_name, '')
    if is_valid_hex_secret(manual_value, expected_length):
        resolved = manual_value.strip().lower()
        source = 'manual_input'
    elif is_valid_hex_secret(existing_value, expected_length):
        resolved = existing_value.strip().lower()
        source = 'existing_env'
    elif AUTO_GENERATE_MISSING_ATTESTATION_KEYS:
        resolved = secrets.token_hex(expected_length // 2)
        source = 'generated_for_session'
        generated_env_vars.append(env_name)
    else:
        resolved = ''
        source = 'missing'

    if resolved:
        os.environ[env_name] = resolved
        resolved_values[env_name] = {
            'source': source,
            'length': len(resolved),
            'masked_value': mask_secret(resolved),
        }
    else:
        invalid_env_vars.append(env_name)

if invalid_env_vars:
    raise RuntimeError(f'attestation 环境变量仍缺失: {invalid_env_vars}')

attestation_info = {
    'saved_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'config_file': str(CONFIG_FILE),
    'generated_env_vars': generated_env_vars,
    'resolved_values': resolved_values,
}
with open(ATTESTATION_INFO_PATH, 'w', encoding='utf-8') as f:
    json.dump(attestation_info, f, indent=2, ensure_ascii=False)

print(f'✓ CONFIG_FILE={CONFIG_FILE}')
print(f'✓ attestation 信息已写出: {ATTESTATION_INFO_PATH}')
for env_name, item in resolved_values.items():
    print(f'  - {env_name}: source={item["source"]}, value={item["masked_value"]}')

## 第 5 步：运行前预检

这个 cell 对应 Colab 上真实 SD3.5 mini-real validation 的最小前置条件检查。只有通过后再执行正式运行。

In [ ]:
import socket

import torch
from huggingface_hub import HfApi

precheck_results = []

def record_check(name: str, ok: bool, detail: str) -> None:
    precheck_results.append({'name': name, 'ok': ok, 'detail': detail})
    print(f'{"✓" if ok else "✗"} {name}: {detail}')

cuda_ok = bool(torch.cuda.is_available())
record_check('CUDA 可用', cuda_ok, torch.cuda.get_device_name(0) if cuda_ok else '未检测到 GPU')

required_paths = [
    CEG_WM_ROOT / 'scripts' / 'run_mini_real_validation.py',
    CEG_WM_ROOT / 'configs' / 'paper_full_cuda_mini_real_validation.yaml',
    CEG_WM_ROOT / 'prompts' / 'paper_small_mini_real.txt',
    CEG_WM_ROOT / 'models' / 'inspyrenet' / 'ckpt_base.pth',
]
for path in required_paths:
    record_check(f'文件存在: {path.name}', path.exists(), str(path))

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    value = os.environ.get(env_name, '').strip().lower()
    ok = len(value) == expected_length and all(ch in '0123456789abcdef' for ch in value)
    record_check(f'attestation 环境变量: {env_name}', ok, f'len={len(value)}')

required_modules = ['yaml', 'huggingface_hub', 'diffusers', 'transformers', 'safetensors']
for module_name in required_modules:
    try:
        __import__(module_name)
        record_check(f'模块可导入: {module_name}', True, 'ok')
    except Exception as exc:
        record_check(f'模块可导入: {module_name}', False, f'{type(exc).__name__}: {exc}')

socket.setdefaulttimeout(15)
try:
    _ = HfApi().model_info('stabilityai/stable-diffusion-3.5-medium')
    record_check('HF 模型可访问', True, 'stabilityai/stable-diffusion-3.5-medium')
except Exception as exc:
    record_check('HF 模型可访问', False, f'{type(exc).__name__}: {exc}')

usage = shutil.disk_usage(str(CEG_WM_ROOT))
record_check('磁盘剩余空间', usage.free / (1024 ** 3) >= 20.0, f'free={usage.free / (1024 ** 3):.1f} GB')

hard_fail_names = {
    'CUDA 可用',
    '文件存在: run_mini_real_validation.py',
    '文件存在: paper_full_cuda_mini_real_validation.yaml',
    '文件存在: paper_small_mini_real.txt',
    '文件存在: ckpt_base.pth',
    'attestation 环境变量: CEG_WM_K_MASTER',
    'attestation 环境变量: CEG_WM_K_PROMPT',
    'attestation 环境变量: CEG_WM_K_SEED',
    '模块可导入: diffusers',
    '模块可导入: transformers',
    'HF 模型可访问',
}
hard_fail = [item for item in precheck_results if item['name'] in hard_fail_names and not item['ok']]
print('\n' + '=' * 80)
if hard_fail:
    print('✗ 预检未通过，请先修复后再执行正式运行')
    for item in hard_fail:
        print(f'  - {item["name"]}: {item["detail"]}')
else:
    print('✓ 预检通过，可以执行 mini-real validation')

## 第 6 步：执行 mini-real validation

本步调用 scripts/run_mini_real_validation.py，并自动落日志与结构化失败摘要。

In [ ]:
print('=' * 80)
print('执行 mini-real validation workflow')
print('=' * 80)

RUN_ROOT = CEG_WM_ROOT / 'outputs' / 'colab_mini_real_validation'
if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'logs').mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = RUN_ROOT / 'artifacts' / 'workflow_acceptance' / 'paper_full_mini_real_validation_summary.json'
WORKFLOW_LOG = RUN_ROOT / 'logs' / 'mini_real_workflow_execution.log'
WORKFLOW_STDOUT_LOG = RUN_ROOT / 'logs' / 'mini_real_workflow_stdout_full.log'
WORKFLOW_STDERR_LOG = RUN_ROOT / 'logs' / 'mini_real_workflow_stderr_full.log'
WORKFLOW_COMMAND_JSON = RUN_ROOT / 'logs' / 'mini_real_workflow_command.json'

command = [
    sys.executable,
    'scripts/run_mini_real_validation.py',
    '--config',
    str(CONFIG_FILE),
    '--run-root',
    str(RUN_ROOT),
]

result = subprocess.run(
    command,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
)

WORKFLOW_STDOUT_LOG.write_text(result.stdout or '', encoding='utf-8')
WORKFLOW_STDERR_LOG.write_text(result.stderr or '', encoding='utf-8')
WORKFLOW_COMMAND_JSON.write_text(
    json.dumps({
        'command': command,
        'cwd': str(CEG_WM_ROOT),
        'return_code': int(result.returncode),
        'summary_path': str(SUMMARY_PATH),
        'stdout_log': str(WORKFLOW_STDOUT_LOG),
        'stderr_log': str(WORKFLOW_STDERR_LOG),
    }, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
WORKFLOW_LOG.write_text(
    'COMMAND:\n' + ' '.join(command) + '\n\nRETURN_CODE:\n' + str(result.returncode) + '\n\nSTDOUT:\n' + (result.stdout or '') + '\n\nSTDERR:\n' + (result.stderr or ''),
    encoding='utf-8',
)

mini_real_summary = {}
if SUMMARY_PATH.exists():
    mini_real_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

print(f'return_code={result.returncode}')
print(f'RUN_ROOT={RUN_ROOT}')
print(f'SUMMARY_PATH={SUMMARY_PATH}')
print(f'WORKFLOW_LOG={WORKFLOW_LOG}')

if result.stdout:
    print('\nSTDOUT tail:')
    print('\n'.join(result.stdout.splitlines()[-40:]))
if result.stderr:
    print('\nSTDERR tail:')
    print('\n'.join(result.stderr.splitlines()[-30:]))

if mini_real_summary:
    print('\nmini-real summary:')
    print(f"  profile_role: {mini_real_summary.get('profile_role')}")
    print(f"  environment_blocked: {mini_real_summary.get('environment_blocked')}")
    print(f"  mini_real_validation_ok: {mini_real_summary.get('mini_real_validation_ok')}")
    print(f"  nearest_failed_stage: {mini_real_summary.get('nearest_failed_stage')}")
    print(f"  issue_count: {mini_real_summary.get('issue_count')}")

## 第 7 步：读取结构化失败摘要

如果运行失败，这一步优先读取新脚本写出的结构化 issues，而不是手动翻全量日志。

In [ ]:
if 'SUMMARY_PATH' not in globals():
    raise RuntimeError('请先执行第 6 步生成 SUMMARY_PATH')

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'未找到 summary: {SUMMARY_PATH}')

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
issues = summary.get('issues', []) if isinstance(summary.get('issues'), list) else []
print('=' * 80)
print('结构化失败摘要')
print('=' * 80)
print(f'nearest_failed_stage={summary.get("nearest_failed_stage")}')
print(f'mini_real_validation_ok={summary.get("mini_real_validation_ok")}')
print(f'issue_count={summary.get("issue_count")}')

for index, issue in enumerate(issues, start=1):
    print('-' * 80)
    print(f'[{index}] issue={issue.get("issue")}')
    print(f'    severity={issue.get("severity")}')
    print(f'    visibility_risk={issue.get("negative_result_visibility_risk")}')
    print(f'    recommended_fix={issue.get("recommended_fix")}')
    evidence = issue.get('evidence', {})
    if isinstance(evidence, dict):
        print(f'    evidence={json.dumps(evidence, ensure_ascii=False)}')

matrix_summary_path = RUN_ROOT / 'outputs' / 'experiment_matrix' / 'artifacts' / 'grid_summary.json'
if matrix_summary_path.exists():
    matrix_summary = json.loads(matrix_summary_path.read_text(encoding='utf-8'))
    print('\nexperiment_matrix 摘要:')
    print(f"  total={matrix_summary.get('total')}")
    print(f"  succeeded={matrix_summary.get('succeeded')}")
    print(f"  failed={matrix_summary.get('failed')}")
    results = matrix_summary.get('results', [])
    if isinstance(results, list):
        failed_items = [item for item in results if isinstance(item, dict) and item.get('status') != 'ok']
        if failed_items:
            print(f"  first_failure_reason={failed_items[0].get('failure_reason')}")

## 第 8 步：导出结果包

这一步会把 mini-real validation 的关键输出打成 zip 并在 Colab 中触发下载。

In [ ]:
import shutil
import zipfile

if 'RUN_ROOT' not in globals():
    raise RuntimeError('请先执行第 6 步生成 RUN_ROOT')

archive_path = WORK_DIR / f'ceg_wm_mini_real_validation_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
included_files = []
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base_dir in [RUN_ROOT / 'records', RUN_ROOT / 'artifacts', RUN_ROOT / 'logs', RUN_ROOT / 'outputs']:
        if not base_dir.exists():
            continue
        for file_path in sorted(base_dir.rglob('*')):
            if not file_path.is_file():
                continue
            arcname = file_path.relative_to(RUN_ROOT).as_posix()
            zf.write(file_path, arcname=arcname)
            included_files.append(arcname)

synced_archive_path = None
drive_export_dir = globals().get('GOOGLE_DRIVE_EXPORT_DIR')
if drive_export_dir is not None:
    drive_export_dir = Path(drive_export_dir)
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    synced_archive_path = drive_export_dir / archive_path.name
    shutil.copy2(archive_path, synced_archive_path)

print(f'✓ 压缩包已生成: {archive_path}')
print(f'✓ 文件数量: {len(included_files)}')
if synced_archive_path is not None:
    print(f'✓ 已同步到 Google Drive: {synced_archive_path}')
else:
    print('ℹ 未检测到 Google Drive 同步目录，跳过同步')

if IN_COLAB:
    try:
        colab_files.download(str(archive_path))
        print('✓ 已触发 Colab 下载')
    except Exception as exc:
        print(f'⚠ 自动下载失败，请手动下载: {archive_path}, reason={exc}')
else:
    print(f'ℹ 非 Colab 环境，请手动获取: {archive_path}')

## 使用建议

- 如果第 5 步预检失败，先修环境，不要直接跑正式验证。
- 如果第 6 步返回码非 0，先看第 7 步的 issues，再决定是否翻完整日志。
- 如果要长期复用 Colab 缓存，把 USE_GOOGLE_DRIVE 改成 True，并把 huggingface_cache 持久化到 Drive。
- 这个 Notebook 的目标是工程级 formal path 核查，不是最终论文级大样本评测。